In [2]:
!pip install -U datasets transformers seqeval evaluate accelerate

import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate

dataset = load_dataset("conll2003", revision="refs/convert/parquet")

label_list = dataset["train"].features["chunk_tags"].feature.names
num_labels = len(label_list)

print("Labels:", label_list)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []

        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

data_collator = DataCollatorForTokenClassification(tokenizer)
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs"
)
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.0 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=c77151d50c74bee37b3fd5690663a3d3bb2a5c5db5eb82bba584b640365c4edd
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
   

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


conll2003/train/0000.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

...

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [3]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [4]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.218838,0.212155,0.904312,0.903047,0.903679


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1756, training_loss=0.3124783565894889, metrics={'train_runtime': 203.7681, 'train_samples_per_second': 68.907, 'train_steps_per_second': 8.618, 'total_flos': 458799186935040.0, 'train_loss': 0.3124783565894889, 'epoch': 1.0})

In [7]:
trainer.state.log_history

[{'loss': 0.5334645385742187,
  'grad_norm': 4.381625175476074,
  'learning_rate': 1.4316628701594535e-05,
  'epoch': 0.2847380410022779,
  'step': 500},
 {'loss': 0.23832130432128906,
  'grad_norm': 2.997011661529541,
  'learning_rate': 8.621867881548975e-06,
  'epoch': 0.5694760820045558,
  'step': 1000},
 {'loss': 0.21883775329589844,
  'grad_norm': 2.047797679901123,
  'learning_rate': 2.927107061503417e-06,
  'epoch': 0.8542141230068337,
  'step': 1500},
 {'eval_loss': 0.21215544641017914,
  'eval_precision': 0.9043116195812904,
  'eval_recall': 0.9030465531981037,
  'eval_f1': 0.903678643645979,
  'eval_runtime': 14.1,
  'eval_samples_per_second': 230.496,
  'eval_steps_per_second': 28.865,
  'epoch': 1.0,
  'step': 1756},
 {'train_runtime': 203.7681,
  'train_samples_per_second': 68.907,
  'train_steps_per_second': 8.618,
  'total_flos': 458799186935040.0,
  'train_loss': 0.3124783565894889,
  'epoch': 1.0,
  'step': 1756},
 {'eval_loss': 0.21215544641017914,
  'eval_precision':

In [8]:
for log in trainer.state.log_history:
    if "eval_f1" in log:
        print(log)

{'eval_loss': 0.21215544641017914, 'eval_precision': 0.9043116195812904, 'eval_recall': 0.9030465531981037, 'eval_f1': 0.903678643645979, 'eval_runtime': 14.1, 'eval_samples_per_second': 230.496, 'eval_steps_per_second': 28.865, 'epoch': 1.0, 'step': 1756}
{'eval_loss': 0.21215544641017914, 'eval_precision': 0.9043116195812904, 'eval_recall': 0.9030465531981037, 'eval_f1': 0.903678643645979, 'eval_runtime': 15.8519, 'eval_samples_per_second': 205.023, 'eval_steps_per_second': 25.675, 'epoch': 1.0, 'step': 1756}
{'eval_loss': 0.21215544641017914, 'eval_precision': 0.9043116195812904, 'eval_recall': 0.9030465531981037, 'eval_f1': 0.903678643645979, 'eval_runtime': 16.9371, 'eval_samples_per_second': 191.887, 'eval_steps_per_second': 24.03, 'epoch': 1.0, 'step': 1756}


In [10]:
from seqeval.metrics import accuracy_score

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": accuracy_score(true_labels, true_predictions)
    }

In [11]:
results = trainer.state.log_history[-1]

print("\nFinal Evaluation Metrics:")
print(f"Precision: {results['eval_precision']:.4f}")
print(f"Recall: {results['eval_recall']:.4f}")
print(f"F1 Score: {results['eval_f1']:.4f}")


Final Evaluation Metrics:
Precision: 0.9043
Recall: 0.9030
F1 Score: 0.9037
